# TPI 1: Adquisición y Análisis Lingüístico de Medios

**Modalidad:** Trabajo Práctico Individual (calificación numérica de 0 a 10).

**Fecha de entrega y exposición:** Jueves 16 de abril. Se realizará de manera expositiva en remoto frente a todo el grupo de estudiantes (aproximadamente 10 minutos por presentación) para que entre quienes participan veamos posibles soluciones.

**Duración estimada de codificación:** 2 horas

**Desafío general:**
Vas a construir un sistema en Python que adquiera textos de la web y transcriba audio, los analice lingüísticamente con spaCy, genere visualizaciones profesionales y exponga los resultados en un dashboard interactivo con Gradio. Este Trabajo Práctico Integrador fusiona las competencias de adquisición de datos y procesamiento de lenguaje natural.

**Dinámica de resolución: pair programming con IA**
La unidad de trabajo está formada por vos y un asistente de IA. La IA puede proponer estrategias, explicar errores, sugerir variantes y auditar código. No reemplaza tu pensamiento ni tu criterio. Toda decisión final, toda justificación y toda versión entregada tienen que estar bajo tu responsabilidad.

---

### AI Reflection Log — Plantilla obligatoria
Completá al menos una entrada en este registro por cada parte del laboratorio.

| Parte | Objetivo de la consulta | Prompt o pedido a la IA | Qué responidó (resumen) | Qué conservaste y por qué | Qué descartaste y por qué | Qué aprendiste |
|---|---|---|---|---|---|---|
| **Parte 1** | | | | | | |
| **Parte 2** | | | | | | |
| **Parte 3** | | | | | | |
| **Parte 4** | | | | | | |
| **Parte 5** | | | | | | |

In [10]:
# PASO 0: Instalación de las librerías necesarias
# Ejecutá esta celda una sola vez.
!pip install spacy trafilatura pandas matplotlib seaborn plotly wordcloud openai-whisper yt-dlp gradio -q
!python -m spacy download es_core_news_lg -q


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_lg')



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
!pip install cloudscraper


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import spacy
import pandas as pd
import trafilatura
import whisper
import json
import gradio as gr
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from collections import Counter
from wordcloud import WordCloud
import os
import trafilatura
import requests

print("Librerías importadas correctamente.")

Librerías importadas correctamente.


## Parte 1: Adquisición Multimodal del Corpus

**Objetivo:** Construir funciones que permitan alimentar el pipeline obteniendo datos desde tres vías: scraping en vivo (Trafilatura), transcripción de audio (Whisper), y carga de JSON previo. Luego, unificarlas en un único DataFrame.

> [!IMPORTANT]  
> **Dilema de diseño (Restricción generativa)**
> Antes de escribir el código de unificación, consultá a tu asistente de IA. Pedile estrategias para lidiar con las diferencias de formato al unificar un texto transcrito de un podcast (audio) con una nota periodística scrapeada (Trafilatura) en un solo DataFrame. 
> Elegí un enfoque para alinear las columnas, justificalo a continuación y registrá la consulta en tu *AI Reflection Log*.

**Escribí tu justificación acá:**
(*Tu respuesta...*)

In [13]:
url1 = "http://forbesargentina.com/negocios/la-primera-entrevista-marcos-galperin-su-sucesor-ariel-szarfsztejn-planes-mercadolibre-2026-n82509"
url2 = "https://www.gutenberg.org/files/58221/58221-h/58221-h.htm"
url3 = "https://www.bbc.com/mundo/noticias-57904843"
urls = [url1,url2,url3]

In [ ]:


# 1.1 Scraping en vivo
def extraer_noticias_web(urls):
    """Extrae el texto de una lista de URLs usando Trafilatura"""
    headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36',
    'Accept-Language': 'es-AR,es;q=0.9,en-US;q=0.8,en;q=0.7',
    }
    index = 0
    responses = []
    try:
        responses = list(map(lambda url : requests.get(url, headers=headers),urls))
        if responses:
            noticias = list(map(lambda respuesta: trafilatura.extract(respuesta.text, include_comments=False, include_images=False,favor_precision=True,include_tables=False),responses))
            if noticias:
                return noticias
            else:
                return "No se pudo extraer texto de la URL."
        else:
            return "No se pudo obtener la URL."
    except Exception as e:
        return f"Ocurrió un error: {e}"
    
    return noticias

In [45]:
import requests
import trafilatura
from bs4 import BeautifulSoup
import json
import os
import re
def extraer_noticias_web(urls):
    """
    Extrae el texto de una lista de URLs usando Trafilatura,
    extrae el h1 con BeautifulSoup, y guarda el resultado como JSON.
    """
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36',
        'Accept-Language': 'es-AR,es;q=0.9,en-US;q=0.8,en;q=0.7',
    }
    
    # 1. Asegurarnos de que exista la carpeta corpus
    os.makedirs("corpus", exist_ok=True)
    
    noticias_guardadas = []
    
    for index, url in enumerate(urls):
        try:
            # 2. Obtener el HTML de la URL
            respuesta = requests.get(url, headers=headers)
            respuesta.raise_for_status() # Verifica que la petición fue exitosa (código 200)
            
            html_content = respuesta.text
            
            # 3. Extraer el h1 usando BeautifulSoup
            soup = BeautifulSoup(html_content, 'html.parser')
            h1_tag = soup.find('h1')
            
            if h1_tag:
                titulo = h1_tag.get_text(strip=True)
            else:
                # Si por algún motivo la página no tiene un H1, le damos un nombre alternativo
                titulo = f"Noticia_sin_H1_{index}"
                
            # 4. Extraer el contenido principal del texto usando Trafilatura
            contenido_noticia = trafilatura.extract(
                html_content, 
                include_comments=False, 
                include_images=False,
                favor_precision=True,
                include_tables=False
            )
            
            if contenido_noticia:
                # 5. Limpiar el título para usarlo como nombre de archivo válido
                nombre_archivo_limpio = re.sub(r'[\\/*?:"<>|]', "", titulo)
                ruta_archivo = os.path.join("corpus", f"{nombre_archivo_limpio}.json")
                
                # 6. Estructurar lo que vamos a guardar en el JSON
                datos_a_guardar = {
                    "url": url,
                    "titulo": titulo,
                    "texto": contenido_noticia
                }
                
                # 7. Guardar en el disco
                with open(ruta_archivo, 'w', encoding='utf-8') as archivo:
                    json.dump(datos_a_guardar, archivo, ensure_ascii=False, indent=4)
                
                noticias_guardadas.append(datos_a_guardar)
                print(f"✅ Éxito: Se guardó '{ruta_archivo}'")
            else:
                print(f"⚠️ Aviso: No se pudo extraer texto principal de {url}")
                
        except Exception as e:
            print(f"❌ Error procesando la URL {url}: {e}")
    
    # Devuelve la lista con los diccionarios por si querés usarlos en otra variable
    return noticias_guardadas
# --- EJEMPLO DE USO ---
# lista_de_urls = ["https://ejemplo.com/noticia1", "https://ejemplo.com/noticia2"]
# mis_noticias = extraer_noticias_web(lista_de_urls)

In [46]:


noticiasGuardadasComoJson = extraer_noticias_web(urls)

✅ Éxito: Se guardó 'corpus\La primera entrevista a Marcos Galperin y su sucesor, Ariel Szarfsztejn. Los planes de Mercado Libre para el 2026.json'
✅ Éxito: Se guardó 'corpus\LA ODISEA.json'
✅ Éxito: Se guardó 'corpus\Jeff Bezos así fue el viaje de 11 minutos al espacio del multimillonario.json'


['Cuando en 1999 un joven emprendedor argentino lanzó lo que sería Mercado Libre ("Mercado Livre" en Brasil o "MELI" en Wall Street) casi nadie imaginaba que ese garage digital terminaría marcando la historia del comercio electrónico en toda Latinoamérica.\nVeintiséis años después, su creador, Marcos Galperin (54), decidió dar un paso al costado como CEO, para asumir la presidencia del directorio. La posta pasará a su socio Ariel Szarfsztejn (44). La decisión, confirmada a partir del 1º de enero de 2026, marca un hito en la historia de la compañía.\nGalperin, quien figura en el Ranking Forbes como el argentino más rico, con una fortuna estimada en US$ 8.500 millones, no deja un liderazgo menor. Szarfsztejn asumirá el mando de una organización con 120.000 empleados y más de 130 millones de clientes mensuales. Y lo hará al frente de una empresa cuyo valor de mercado supera los US$ 100.000 millones, la más valiosa de toda Latinoamérica.\nEn la entrevista con Exame.com, Galperin describió 

In [ ]:
"""def youtube_a_texto(url: str, modelo_whisper: str = "small", idioma: str = "es", output_dir: str = "corpus") -> dict:
    # Pipeline completo: URL de YouTube -> audio -> transcripcion -> texto
    import json
    import os
    import re
    import shutil
    import whisper
    import yt_dlp

    os.makedirs(output_dir, exist_ok=True)

    if "preparar_ffmpeg" in globals():
        _, ffmpeg_dir, _ = preparar_ffmpeg()
    else:
        ruta_ffmpeg = os.environ.get("FFMPEG_PATH") or shutil.which("ffmpeg")
        if not ruta_ffmpeg or not os.path.exists(ruta_ffmpeg):
            raise FileNotFoundError("No se encontro ffmpeg para ejecutar el pipeline completo.")
        ffmpeg_dir = os.path.dirname(ruta_ffmpeg)
        if ffmpeg_dir not in os.environ.get("PATH", ""):
            os.environ["PATH"] = ffmpeg_dir + os.pathsep + os.environ.get("PATH", "")

    ydl_opts = {
        "format": "bestaudio/best",
        "noplaylist": True,
        "ffmpeg_location": ffmpeg_dir,
        "postprocessors": [
            {
                "key": "FFmpegExtractAudio",
                "preferredcodec": "mp3",
                "preferredquality": "192",
            }
        ],
        "outtmpl": f"{output_dir}/%(title)s.%(ext)s",
    }

    if "sanitizar_nombre_archivo" in globals():
        sanitizar = sanitizar_nombre_archivo
    else:
        def sanitizar(ruta_o_nombre: str, max_len: int = 80) -> str:
            nombre = os.path.splitext(os.path.basename(ruta_o_nombre))[0]
            invalidos = set('<>:/\\|?*') | {chr(34)}
            nombre = ''.join('_' if caracter in invalidos or ord(caracter) < 32 else caracter for caracter in nombre)
            nombre = re.sub(r'\s+', ' ', nombre).strip().rstrip('. ')
            return (nombre[:max_len] or 'transcripcion').strip()

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)
        titulo = info["title"]
        duracion = info["duration"]
        audio_generado = os.path.abspath(ydl.prepare_filename(info).rsplit(".", 1)[0] + ".mp3")

    print(f"Audio descargado: {titulo} ({duracion} segundos)")

    cache_modelos = globals().setdefault("_MODELOS_WHISPER", {})
    if modelo_whisper not in cache_modelos:
        cache_modelos[modelo_whisper] = whisper.load_model(modelo_whisper)
    modelo = cache_modelos[modelo_whisper]
    resultado_local = modelo.transcribe(audio_generado, language=idioma, fp16=False)
    print(f"Transcripcion completa: {len(resultado_local['segments'])} segmentos")

    nombre_base = sanitizar(audio_generado)
    ruta_txt = os.path.join(output_dir, f"{nombre_base}.txt")
    with open(ruta_txt, "w", encoding="utf-8") as archivo_txt:
        archivo_txt.write(resultado_local["text"])

    ruta_json = os.path.join(output_dir, f"{nombre_base}.json")
    with open(ruta_json, "w", encoding="utf-8") as archivo_json:
        json.dump(
            {
                "fuente": url,
                "titulo": titulo,
                "duracion_segundos": duracion,
                "idioma": resultado_local["language"],
                "texto": resultado_local["text"],
                "segmentos": [
                    {"inicio": segmento["start"], "fin": segmento["end"], "texto": segmento["text"].strip()}
                    for segmento in resultado_local["segments"]
                ],
            },
            archivo_json,
            ensure_ascii=False,
            indent=2,
        )

    return {
        "titulo": titulo,
        "duracion": duracion,
        "texto": resultado_local["text"],
        "segmentos": resultado_local["segments"],
        "archivos_generados": [audio_generado, ruta_txt, ruta_json],
    }


# Uso sugerido:
# corpus = youtube_a_texto("https://www.youtube.com/watch?v=yX2tPSjBoeU")
# print(corpus["texto"][:500])

CHOCLASO MAL
""" 

In [ ]:
"""def main():
    urls = [
    "https://www.youtube.com/watch?v=yX2tPSjBoeU",
    "https://www.youtube.com/watch?v=ruepxLoEwoo",
    "https://www.youtube.com/watch?v=WqXr0AujesY",
    ]

    resultados = []
    errores = []

    for indice, url in enumerate(urls, start=1):
        print("\n" + "=" * 60)
        print(f"Procesando {indice}/{len(urls)}: {url}")
        print("=" * 60)
        try:
            resultado_lote = youtube_a_texto(url, modelo_whisper="small")
            resultados.append(resultado_lote)
            print(f"OK: {resultado_lote['titulo']} - {len(resultado_lote['texto'])} caracteres")
        except Exception as error:
            errores.append({"url": url, "error": str(error)})
            print(f"ERROR: {error}")

    print(f"\nResumen: {len(resultados)} exitosos, {len(errores)} errores")

    corpus_total = "\n\n---\n\n".join(resultado_lote["texto"] for resultado_lote in resultados)
    print(f"Corpus total: {len(corpus_total)} caracteres, ~{len(corpus_total.split())} palabras")
    return corpus_total




if __name__ == "__main__":
    main()
    """


Procesando 1/3: https://www.youtube.com/watch?v=yX2tPSjBoeU
[youtube] Extracting URL: https://www.youtube.com/watch?v=yX2tPSjBoeU
[youtube] yX2tPSjBoeU: Downloading webpage


[youtube] yX2tPSjBoeU: Downloading android vr player API JSON
[info] yX2tPSjBoeU: Downloading 1 format(s): 251
[download] Destination: corpus\¿Cuál es el motor de la ciencia？ ｜ Mario Ponce ｜ TEDxLINTAC Youth.webm
[download] 100% of   13.47MiB in 00:00:01 at 10.25MiB/s    
[ExtractAudio] Destination: corpus\¿Cuál es el motor de la ciencia？ ｜ Mario Ponce ｜ TEDxLINTAC Youth.mp3
Deleting original file corpus\¿Cuál es el motor de la ciencia？ ｜ Mario Ponce ｜ TEDxLINTAC Youth.webm (pass -k to keep)
Audio descargado: ¿Cuál es el motor de la ciencia? | Mario Ponce | TEDxLINTAC Youth (1173 segundos)
Transcripcion completa: 219 segmentos
OK: ¿Cuál es el motor de la ciencia? | Mario Ponce | TEDxLINTAC Youth - 16322 caracteres

Procesando 2/3: https://www.youtube.com/watch?v=ruepxLoEwoo
[youtube] Extracting URL: https://www.youtube.com/watch?v=ruepxLoEwoo
[youtube] ruepxLoEwoo: Downloading webpage


[youtube] ruepxLoEwoo: Downloading android vr player API JSON
[info] ruepxLoEwoo: Downloading 1 format(s): 251
[download] Destination: corpus\La ciencia que te rodea y que no ves ｜ César Sobrero ｜ TEDxCONICETRosario.webm
[download] 100% of    9.77MiB in 00:00:01 at 9.19MiB/s     
[ExtractAudio] Destination: corpus\La ciencia que te rodea y que no ves ｜ César Sobrero ｜ TEDxCONICETRosario.mp3
Deleting original file corpus\La ciencia que te rodea y que no ves ｜ César Sobrero ｜ TEDxCONICETRosario.webm (pass -k to keep)
Audio descargado: La ciencia que te rodea y que no ves | César Sobrero | TEDxCONICETRosario (740 segundos)
Transcripcion completa: 149 segmentos
OK: La ciencia que te rodea y que no ves | César Sobrero | TEDxCONICETRosario - 10381 caracteres

Procesando 3/3: https://www.youtube.com/watch?v=WqXr0AujesY
[youtube] Extracting URL: https://www.youtube.com/watch?v=WqXr0AujesY
[youtube] WqXr0AujesY: Downloading webpage


[youtube] WqXr0AujesY: Downloading android vr player API JSON
[info] WqXr0AujesY: Downloading 1 format(s): 251
[download] Destination: corpus\Ciencia Clara： Rompiendo Barreras. ｜ Itzel Martinez ｜ TEDxPaseoUsumacintaAve.webm
[download] 100% of    5.78MiB in 00:00:00 at 10.42MiB/s    
[ExtractAudio] Destination: corpus\Ciencia Clara： Rompiendo Barreras. ｜ Itzel Martinez ｜ TEDxPaseoUsumacintaAve.mp3
Deleting original file corpus\Ciencia Clara： Rompiendo Barreras. ｜ Itzel Martinez ｜ TEDxPaseoUsumacintaAve.webm (pass -k to keep)
Audio descargado: Ciencia Clara: Rompiendo Barreras. | Itzel Martinez | TEDxPaseoUsumacintaAve (390 segundos)
Transcripcion completa: 44 segmentos
OK: Ciencia Clara: Rompiendo Barreras. | Itzel Martinez | TEDxPaseoUsumacintaAve - 4263 caracteres

Resumen: 3 exitosos, 0 errores
Corpus total: 30980 caracteres, ~5461 palabras


In [47]:
import json
import os
import re
import whisper
import yt_dlp

def sanitizar_nombre(nombre: str) -> str:
    """Limpia un texto eliminando caracteres inválidos para usarlo como nombre de archivo."""
    return re.sub(r'[\\/*?:"<>|]', "", nombre).strip()

def procesar_videos_youtube(urls: list, modelo_whisper: str = "small", idioma: str = "es", output_dir: str = "corpus"):
    """
    Recibe una lista de URLs, descarga el audio, lo transcribe con Whisper
    y guarda un JSON (y el MP3) nombrados con el título original del video.
    """
    os.makedirs(output_dir, exist_ok=True)
    resultados = []
    
    # 1. Cargamos el modelo Whisper una sola vez al principio (Optimiza MUCHO el tiempo)
    print(f"Cargando modelo Whisper '{modelo_whisper}'... (Esto puede demorar unos segundos)")
    modelo = whisper.load_model(modelo_whisper)
    
    for indice, url in enumerate(urls, start=1):
        print(f"\n[{indice}/{len(urls)}] Procesando: {url}")
        
        try:
            # 2. Configuración de yt_dlp para descargar solo audio como MP3
            ydl_opts = {
                "format": "bestaudio/best",
                "noplaylist": True,
                "postprocessors": [{
                    "key": "FFmpegExtractAudio",
                    "preferredcodec": "mp3",
                    "preferredquality": "192",
                }],
                # Guardamos temporalmente usando el ID del video para no tener problemas de caracteres en la descarga
                "outtmpl": f"{output_dir}/%(id)s.%(ext)s", 
                "quiet": True,
                "no_warnings": True
            }
            
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                info = ydl.extract_info(url, download=True)
                titulo = info.get("title", f"Video_sin_titulo_{indice}")
                duracion = info.get("duration", 0)
                
                # Esta es la ruta temporal que le asignó yt-dlp
                audio_temp_path = os.path.join(output_dir, f"{info['id']}.mp3")
            
            print(f"✅ Audio descargado. Transcribiendo '{titulo}'...")
            
            # 3. Transcribir con Whisper
            resultado_whisper = modelo.transcribe(audio_temp_path, language=idioma, fp16=False)
            
            # 4. Limpiar el título para guardar los archivos definitivos
            nombre_limpio = sanitizar_nombre(titulo)
            ruta_json = os.path.join(output_dir, f"{nombre_limpio}.json")
            ruta_audio_final = os.path.join(output_dir, f"{nombre_limpio}.mp3")
            
            # 5. Generar y guardar el JSON
            datos_json = {
                "fuente": url,
                "titulo": titulo,
                "duracion_segundos": duracion,
                "idioma": resultado_whisper.get("language", idioma),
                "texto": resultado_whisper["text"],
                "segmentos": [
                    {"inicio": seg["start"], "fin": seg["end"], "texto": seg["text"].strip()}
                    for seg in resultado_whisper.get("segments", [])
                ]
            }
            
            with open(ruta_json, "w", encoding="utf-8") as archivo_json:
                json.dump(datos_json, archivo_json, ensure_ascii=False, indent=4)
                
            # 6. Renombrar el MP3 para que tenga el mismo nombre hermoso que el JSON
            if os.path.exists(audio_temp_path):
                if os.path.exists(ruta_audio_final): # Si ya existía uno viejo, lo borramos
                    os.remove(ruta_audio_final)
                os.rename(audio_temp_path, ruta_audio_final)
                
            resultados.append(datos_json)
            print(f"✅ Completado. Guardado en '{ruta_json}'")
            
        except Exception as e:
            print(f"❌ Error al procesar {url}: {e}")
            
    return resultados




In [48]:
# ==========================================
# EJEMPLO DE USO (Lo que iría en tu main)
# ==========================================

def main():
    urls_youtube = [
        "https://www.youtube.com/watch?v=8TGyUgUpmvk",
        "https://www.youtube.com/watch?v=wdvrPU2rgeI",
        "https://www.youtube.com/shorts/bGdfMARFhPA",
    ]
    
    print("Iniciando pipeline de YouTube a Texto...")
    
    # Solo llamamos a la función genérica
    mis_transcripciones = procesar_videos_youtube(urls_youtube)
    
    print("\n" + "=" * 50)
    print(f"Proceso finalizado. Se extrajeron {len(mis_transcripciones)} videos.")
    print("=" * 50)

if __name__ == "__main__":
    main()


Iniciando pipeline de YouTube a Texto...
Cargando modelo Whisper 'small'... (Esto puede demorar unos segundos)

[1/3] Procesando: https://www.youtube.com/watch?v=8TGyUgUpmvk
✅ Audio descargado. Transcribiendo 'Charla motivacional de Marcelo Bielsa ¡INCREÍBLE!'...
✅ Completado. Guardado en 'corpus\Charla motivacional de Marcelo Bielsa ¡INCREÍBLE!.json'

[2/3] Procesando: https://www.youtube.com/watch?v=wdvrPU2rgeI
✅ Audio descargado. Transcribiendo 'El papel de un líder es influir, transmitir y contagiar'...
✅ Completado. Guardado en 'corpus\El papel de un líder es influir, transmitir y contagiar.json'

[3/3] Procesando: https://www.youtube.com/shorts/bGdfMARFhPA
✅ Audio descargado. Transcribiendo 'Una gran LECCIÓN de VIDA #motivacion #exito #consejos #desarrollopersonal'...
✅ Completado. Guardado en 'corpus\Una gran LECCIÓN de VIDA #motivacion #exito #consejos #desarrollopersonal.json'

Proceso finalizado. Se extrajeron 3 videos.


In [49]:
import os
import glob
import json

def cargar_json_desde_carpeta(carpeta="corpus"):
    """
    Busca todos los archivos .json en la carpeta especificada 
    y los carga en una lista de diccionarios usando la librería nativa json.
    """
    # 1. glob.glob busca todos los archivos que coincidan con la extensión .json
    patron_busqueda = os.path.join(carpeta, "*.json")
    rutas_json = glob.glob(patron_busqueda)
    
    if not rutas_json:
        print(f"⚠️ No se encontraron archivos JSON en la carpeta '{carpeta}'.")
        return []
        
    print(f"Se encontraron {len(rutas_json)} archivos. Cargando...")
    
    corpus = []
    
    # 2. Iteramos sobre cada ruta encontrada y abrimos el archivo
    for ruta in rutas_json:
        try:
            with open(ruta, 'r', encoding='utf-8') as archivo:
                datos_json = json.load(archivo)
                corpus.append(datos_json)
                
        except Exception as e:
            print(f"❌ Error al intentar cargar {ruta}: {e}")
            
    return corpus





In [51]:
# ==========================================
# EJEMPLO DE USO (Lo que iría en tu main)
# ==========================================
def main():
    # Solo le pasamos el nombre de la carpeta, la función hace el resto
    mi_corpus = cargar_json_desde_carpeta("corpus")
    
    print("\n" + "="*50)
    print(f"¡Carga finalizada! Tenés {len(mi_corpus)} documentos en memoria.")
    
    # Mostrar el título del primer documento como prueba
    if mi_corpus:
        index = 0
        for mi in mi_corpus:
            print(f"Primer documento cargado: {mi_corpus[index].get('titulo', 'Sin título')}")
            index = index + 1
if __name__ == "__main__":
    main()

Se encontraron 9 archivos. Cargando...

¡Carga finalizada! Tenés 9 documentos en memoria.
Primer documento cargado: Charla motivacional de Marcelo Bielsa ¡INCREÍBLE!
Primer documento cargado: Ciencia Clara: Rompiendo Barreras. | Itzel Martinez | TEDxPaseoUsumacintaAve
Primer documento cargado: El papel de un líder es influir, transmitir y contagiar
Primer documento cargado: Jeff Bezos: así fue el viaje de 11 minutos al espacio del multimillonario
Primer documento cargado: La ciencia que te rodea y que no ves | César Sobrero | TEDxCONICETRosario
Primer documento cargado: LA ODISEA
Primer documento cargado: La primera entrevista a Marcos Galperin y su sucesor, Ariel Szarfsztejn. Los planes de Mercado Libre para el 2026
Primer documento cargado: Una gran LECCIÓN de VIDA #motivacion #exito #consejos #desarrollopersonal
Primer documento cargado: ¿Cuál es el motor de la ciencia? | Mario Ponce | TEDxLINTAC Youth


In [20]:
# 1.4 Consolidación
def unificar_corpus(datos_web, datos_audio, datos_json):
    """Unifica las tres fuentes en un DataFrame con columnas estándar"""
    # PASO 4: Implementá la estrategia que decidiste en el dilema de diseño.
    # El DataFrame final debería tener al menos: 'titulo_o_fuente', 'texto', 'origen' ('web', 'audio', 'json')
    
    df_unificado = pd.DataFrame()
    return df_unificado

# ---- Espacio para pruebas ----
# Probá tus funciones acá con al menos 1 url web y 1 video corto.
# df_corpus = unificar_corpus(...)

In [54]:
import pandas as pd

def unificar_corpus(datos_web, datos_audio, datos_json):
    """
    Unifica las tres fuentes (listas de diccionarios) en un único 
    DataFrame de Pandas con columnas estandarizadas.
    """
    filas_estandarizadas = []
    
    # 1. Procesar datos de extracción Web
    if datos_web:
        for doc in datos_web:
            filas_estandarizadas.append({
                # Si no tiene título, usamos la URL, y si no, un texto por defecto
                'titulo_o_fuente': doc.get('titulo', doc.get('url', 'Web sin título')),
                'texto': doc.get('texto', doc.get('contenido', '')),
                'origen': 'web'
            })
            
    # 2. Procesar datos de YouTube/Audio
    if datos_audio:
        for doc in datos_audio:
            filas_estandarizadas.append({
                'titulo_o_fuente': doc.get('titulo', doc.get('fuente', 'Audio sin título')),
                'texto': doc.get('texto', ''),
                'origen': 'audio'
            })
            
    # 3. Procesar datos cargados de archivos JSON locales
    if datos_json:
        for doc in datos_json:
            # Los JSON podrían ser heterogéneos, así que buscamos varias claves posibles
            titulo = doc.get('titulo', doc.get('title', doc.get('url', doc.get('fuente', 'JSON sin título'))))
            texto = doc.get('texto', doc.get('contenido', ''))
            
            filas_estandarizadas.append({
                'titulo_o_fuente': titulo,
                'texto': texto,
                'origen': 'json'
            })
            
    # 4. Convertir la lista de diccionarios a DataFrame
    df_unificado = pd.DataFrame(filas_estandarizadas)
    
    # OPCIONAL: Filtrar documentos que no tengan texto (si falló una extracción)
    if not df_unificado.empty:
        df_unificado = df_unificado[df_unificado['texto'].str.strip() != '']
    
    # Reiniciar el índice para que quede prolijo (0, 1, 2, ...)
    df_unificado.reset_index(drop=True, inplace=True)
    
    return df_unificado

# ==========================================
# ESPACIO PARA PRUEBAS (Integración)
# ==========================================
# (Suponiendo que tenés las funciones anteriores definidas arriba)

def main_pruebas():
    print("Iniciando extracción y unificación...")
    
    # 1. Probar Web (1 URL)
    urls_web = ["https://es.wikipedia.org/wiki/Procesamiento_de_lenguajes_naturales"]
    print("\n--- Extrayendo Web ---")
    datos_web = extraer_noticias_web(urls_web)
    
    # 2. Probar Audio (1 Video corto)
    urls_youtube = ["https://www.youtube.com/watch?v=yX2tPSjBoeU"]
    print("\n--- Extrayendo Audio ---")
    datos_audio = procesar_videos_youtube(urls_youtube)
    
    # 3. Probar JSON locales
    print("\n--- Cargando JSONs ---")
    datos_json = cargar_json_desde_carpeta("corpus")
    
    # 4. Consolidar todo en el DataFrame
    print("\n--- Unificando Corpus ---")
    df_corpus = unificar_corpus(datos_web, datos_audio, datos_json)
    
    print("\n¡Proceso Terminado! Vista previa del DataFrame:")
    # Muestra las primeras 5 filas y las columnas
    display(df_corpus.head()) 
    
    # Opcional: mostrar la distribución de orígenes
    print("\nDistribución por origen:")
    print(df_corpus['origen'].value_counts())

# Ejecutar la prueba 
main_pruebas()


Iniciando extracción y unificación...

--- Extrayendo Web ---
✅ Éxito: Se guardó 'corpus\Procesamiento de lenguajes naturales.json'

--- Extrayendo Audio ---
Cargando modelo Whisper 'small'... (Esto puede demorar unos segundos)

[1/1] Procesando: https://www.youtube.com/watch?v=yX2tPSjBoeU
✅ Audio descargado. Transcribiendo '¿Cuál es el motor de la ciencia? | Mario Ponce | TEDxLINTAC Youth'...
✅ Completado. Guardado en 'corpus\¿Cuál es el motor de la ciencia  Mario Ponce  TEDxLINTAC Youth.json'

--- Cargando JSONs ---
Se encontraron 2 archivos. Cargando...

--- Unificando Corpus ---

¡Proceso Terminado! Vista previa del DataFrame:


,titulo_o_fuente,texto,origen
0,Procesamiento de lenguajes naturales,El procesamiento de(l) lenguaje natural o de l...,web
1,¿Cuál es el motor de la ciencia? | Mario Ponce...,¿Qué mueve a la ciencia? ¿Cuál es el motor de...,audio
2,Procesamiento de lenguajes naturales,El procesamiento de(l) lenguaje natural o de l...,json
3,¿Cuál es el motor de la ciencia? | Mario Ponce...,¿Qué mueve a la ciencia? ¿Cuál es el motor de...,json



Distribución por origen:
origen
json     2
web      1
audio    1
Name: count, dtype: int64


In [ ]:
"""def main():
    rutas_json = []

    print(rutas_json)
    corpus = cargar_json_previo(rutas_json)
    print(len(corpus))



if __name__ == "__main__":
    main()"""

['corpus/¿Cuál es el motor de la ciencia？ ｜ Mario Ponce ｜ TEDxLINTAC Youth.json', 'corpus/Ciencia Clara： Rompiendo Barreras. ｜ Itzel Martinez ｜ TEDxPaseoUsumacintaAve.json', 'corpus/La ciencia que te rodea y que no ves ｜ César Sobrero ｜ TEDxCONICETRosario.json']
3


> **Pausa de auditoría:**
> Revisá tu DataFrame consolidado (`df_corpus.head()`). ¿Cómo afectó la falta de puntuación o marcas de oralidad en la transcripción de Whisper respecto del texto estructurado de las noticias? Revisá las columnas generadas. ¿Perdiste información contextual al unificarlas?

## Parte 2: Análisis Lingüístico con spaCy

**Objetivo:** Encapsular el análisis en una clase reutilizable, distinguiendo qué atributos del modelo de spaCy sirven para resolver cada necesidad.

> [!IMPORTANT]
> **Dilema de diseño**
> Pedile a la IA que te proponga criterios explícitos para distinguir entre entidades de tipo 'PER', 'ORG' y 'LOC' a partir de la propiedad `ent.label_` de spaCy. Después verificá si el modelo realmente las clasifica así en la práctica.
> Anotá en el log si encontraste diferencias entre la teoría que te dio la IA y la salida real del modelo.

In [21]:
class AnalizadorCorpus:
    def __init__(self, df, modelo_spacy="es_core_news_lg"):
        self.df = df
        print("Cargando modelo de lenguaje...")
        self.nlp = spacy.load(modelo_spacy)
        
        # Procesamos la columna 'texto' al instanciar la clase
        print("Procesando los textos con spaCy...")
        # PASO 1: Creá una nueva columna en el DataFrame llamada 'doc' que contenga el objeto procesado por self.nlp()
        # self.df['doc'] = ...
        
    def extraer_entidades(self):
        """Devuelve las entidades agrupadas por tipo, contabilizando total de apariciones"""
        # PASO 2: Recorré los 'doc' del DataFrame y armá un diccionario o lista con las entidades halladas.
        pass

    def extraer_verbos_principales(self, n=15):
        """Devuelve los 'n' verbos lematizados más frecuentes de todo el corpus"""
        # PASO 3: Filtrá tokens que sean VERB y no sean stopwords, extraé su lema y contalos.
        pass

    def extraer_palabras_clave(self, n=20):
        """Devuelve sustantivos y nombres propios lematizados y filtrados"""
        # PASO 4: Implementá una lógica superior a la del Lab 009 (donde usamos stopwords crudas).
        # Filtrá por categorías gramaticales relevantes (NOUN, PROPN, ADJ) omitiendo puntuación y stopwords.
        pass
        
    def estadisticas_corpus(self):
        """Genera un diccionario con métricas generales del corpus"""
        # PASO 5: Calculá total de tokens, tamaño del vocabulario único (lemas) y cantidad de oraciones.
        pass

# ---- Espacio para pruebas ----
# analizador = AnalizadorCorpus(df_corpus)
# print(analizador.estadisticas_corpus())

> **Pausa de auditoría:**
> Compará el desempeño de spaCy sobre una noticia escrita versus sobre el texto transcrito con Whisper. ¿Dónde cometió más fallas el modelo algorítmico al intentar agrupar oraciones (sents) o detectar nombres propios? ¿Por qué creés que se da este fenómeno?

## Parte 3: Visualización Profesional

**Objetivo:** Aplicar principios de Data-Ink Ratio, accesibilidad y jerarquía visual para comunicar hallazgos efectivamente, en lugar de imprimir datos planos.

> [!IMPORTANT]
> **Dilema de diseño**
> Consultá a la IA: ¿conviene usar un *WordCloud* o un *Barplot* para mostrar frecuencias de palabras clave en un informe dirigido a toma de decisiones? Justificá tu elección aplicando el principio de Data-Ink Ratio.

**Escribí tu justificación acá:**
(*Tu respuesta...*)

In [22]:
# Configuración base de accesibilidad visual
sns.set_theme(style="ticks", palette="colorblind", font_scale=1.1)
COLOR_ACENTO = sns.color_palette("colorblind")[0]
COLOR_BASE = '#b0b0b0'

def visualizar_origen(df):
    """Genera un barplot con el origen de los datos o las secciones"""
    # PASO 1: Generá un barplot horizontal orientado a objetos (fig, ax) usando Seaborn.
    # Aplicá el COLOR_ACENTO a la barra principal (la de mayor count).
    # Despintá los bordes molestos con sns.despine()
    pass

def visualizar_palabras_clave_lollipop(palabras_clave):
    """Genera un Lollipop Chart de las palabras clave lematizadas"""
    # PASO 2: Construí el gráfico estructurado (Lollipop) para las palabras clave extraídas en la Parte 2.
    # Recordá que el lollipop se arma combinando ax.hlines y ax.plot.
    pass

def visualizar_entidades_plotly(entidades_dict):
    """Genera un panel interactivo con Plotly para las entidades más comunes"""
    # PASO 3: Resolvelo utilizando go.Bar y devolvé el objeto figura (fig) de Plotly
    # para usarlo posteriormente en Gradio.
    pass

> **Pausa de auditoría:**
> Revisá tu visualización. ¿Es accesible? El uso de la paleta 'colorblind' asegura que ciertos grados de daltonismo no impidan la lectura cromática, pero ¿el tamaño de fuente y la proporción de la figura se leen correctamente sin forzar la vista? ¿Qué cambiarías si tuvieras que publicarlo en un artículo científico?

## Parte 4: Pipeline Integrado (Orquestación)

**Objetivo:** Orquestar los componentes desarrollados en un flujo lógico unificado y persistir los hallazgos en formato estructurado. Todo sistema analítico debe poder guardar su estado final de forma estática.

In [23]:
class PipelineMediatico:
    def __init__(self, urls_web=None, url_audio=None, ruta_json=None):
        self.urls_web = urls_web or []
        self.url_audio = url_audio
        self.ruta_json = ruta_json
        self.df = None
        self.analizador = None
        
    def ejecutar_pipeline(self):
        """Orquesta la adquisición, unificación y análisis"""
        # PASO 1: Orquestá las llamadas a las funciones de la Parte 1.
        
        # PASO 2: Instanciá AnalizadorCorpus y derivale el DataFrame resultante para procesar.

        print("Pipeline ejecutado exitosamente.")
        
    def generar_reporte_y_exportar(self, ruta_csv="corpus_resultante.csv", ruta_json="estadisticas.json"):
        """Exporta el dataframe y un JSON analítico"""
        # PASO 3: Persistí self.df como CSV.
        # ¡OJO! La columna 'doc' de spaCy no es serializable, deberías dropearla o extraer sus textos antes de guardar.
        
        # PASO 4: Persistí las estadísticas y el diccionario de entidades devueltas por el Analizador como JSON local.
        pass

# ---- Espacio para pruebas ----
# pipeline = PipelineMediatico(urls_web=["..."], url_audio="...")
# pipeline.ejecutar_pipeline()
# pipeline.generar_reporte_y_exportar()

> **Pausa de auditoría:**
> Imaginá que un equipo de periodismo de datos de tu facultad te pide el corpus procesado. ¿Qué información necesitaban ellos en el CSV plano versus qué preferiste consolidar en el JSON jerárquico? Pensá por qué separamos esas dos naturalezas de exportación y registralo.

## Parte 5: Dashboard Interactivo con Gradio

**Objetivo:** Diseñar una interfaz interactiva que reaccione dinámicamente frente al usuario, conectando el backend analítico con componentes preconstruidos de frontend.

> [!IMPORTANT]
> **Dilema de diseño**
> ¿Qué componentes elegirías para cada salida? Pedile a la IA tres layouts de estructura (por ejemplo: Pestañas vs. Columna vertical vs. Acordeón) para tu dashboard. Elegí el que consideres mejor para la experiencia de lectura evaluativa y descartá explícitamente los otros dos argumentando tu postura técnica.

**Escribí tu justificación acá:**
(*Tu respuesta...*)

In [24]:
# PASO 1: Diseñá el bloque principal de gr.Blocks() interactuando con los métodos de la clase AnalizadorCorpus.
# Sugerencia: Utilizá pestañas (gr.Tab) para separar "Métricas Generales" de "Filtros e Interacción".

with gr.Blocks(theme=gr.themes.Soft()) as dashboard_medios:
    gr.Markdown("# Explorador de Agenda Mediática")
    
    with gr.Tab("Panorama y Métricas"):
        # Incluí acá la visualización de frecuencias y orígenes, acompañando un gr.DataFrame con métricas generales.
        pass
        
    with gr.Tab("Explorador de Entidades"):
        # Desarrollá un textbox para ingresar una entidad y un botón que dispare
        # un filtrado, mostrando sólo las oraciones dentro de los textos donde se mencionó dicha entidad.
        pass

# Descomentá la siguiente línea cuando el bloque esté terminado
# dashboard_medios.launch()

C:\Users\Emilio\AppData\Local\Temp\ipykernel_23844\2256453203.py:4: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as dashboard_medios:


---
## Cierre Formal y Checklist de Entrega

1. ¿Corriste el pipeline de principio a fin, comprobando que las funciones se anidan y comparten el DataFrame correctamente?
2. ¿Tu *AI Reflection Log* evidencia que cuestionaste a la IA y al modelo algorítmico, o todas tus celdas dicen "me devolvió un código y lo usé"?
3. ¿Revisaste el impacto visual de los gráficos garantizando que minimizan la "tinta algorítmica" (Data-Ink Ratio)?
4. ¿Justificaste tus decisiones de arquitectura técnica en el código de orquestación y exportación?

Si respondiste positivamente: felicitaciones, completaste el **TPI 1** demostrando un uso constructivo de la IA, asumiendo un rol profesional capaz de dirigir la automatización de forma estratégica e informada.